In [1]:
from pathlib import Path
import pandas as pd
import networkx as nx

# Works when the notebook runs from the project or notebooks folder
ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"

hubs = pd.read_csv(PROCESSED / "hubs_threshold3.csv")
lanes = pd.read_csv(PROCESSED / "lanes.csv")
legs = pd.read_csv(PROCESSED / "legs.csv")

assert hubs["hub"].is_unique, "Duplicate hub IDs found."

# Accepted city-level coordinates, not exact warehouse locations
hubs["located"] = (
    hubs["match"].eq("city+state")
    & pd.to_numeric(hubs["latitude"], errors="coerce").between(-90, 90)
    & pd.to_numeric(hubs["longitude"], errors="coerce").between(-180, 180)
)

# Keep lanes observed on at least three distinct trips
lanes_3 = lanes.loc[lanes["trip_count"] >= 3].copy()

G = nx.from_pandas_edgelist(
    lanes_3,
    source="source_center",
    target="destination_center",
    edge_attr=["trip_count", "leg_count"],
    create_using=nx.DiGraph()
)

print("Project:", ROOT)
print("Journey legs:", len(legs))
print("Threshold-3 hubs:", G.number_of_nodes())
print("Threshold-3 lanes:", G.number_of_edges())
print("Hubs with accepted coordinates:", hubs["located"].sum())

Project: c:\local\supplyshield
Journey legs: 26369
Threshold-3 hubs: 1434
Threshold-3 lanes: 1945
Hubs with accepted coordinates: 973


In [2]:
# Find the largest group connected when lane direction is ignored
largest_group = max(
    nx.weakly_connected_components(G),
    key=lambda group: (len(group), tuple(sorted(group)))
)

H = G.subgraph(largest_group).copy()
U = H.to_undirected()

# Traffic touching each hub
traffic = {
    hub: (
        sum(d["leg_count"] for _, _, d in H.in_edges(hub, data=True))
        + sum(d["leg_count"] for _, _, d in H.out_edges(hub, data=True))
    )
    for hub in H
}

# Stable ordering makes the result reproducible
seed = sorted(H, key=lambda hub: (-traffic[hub], hub))[0]
selected = {seed}
selection_order = [seed]

while len(selection_order) < min(150, len(H)):
    neighbours = set()
    for hub in selected:
        neighbours.update(U.neighbors(hub))
    candidates = neighbours - selected

    def added_legs(hub):
        incoming = sum(
            H[other][hub]["leg_count"]
            for other in H.predecessors(hub)
            if other in selected
        )
        outgoing = sum(
            H[hub][other]["leg_count"]
            for other in H.successors(hub)
            if other in selected
        )
        return incoming + outgoing

    chosen = sorted(
        candidates,
        key=lambda hub: (-added_legs(hub), -traffic[hub], hub)
    )[0]

    selected.add(chosen)
    selection_order.append(chosen)

located_ids = set(hubs.loc[hubs["located"], "hub"])
core_candidates = {}
results = []

for size in [50, 100, 150]:
    if size > len(selection_order):
        continue

    core_ids = set(selection_order[:size])
    core = H.subgraph(core_ids).copy()

    # Only legs belonging to the retained threshold-3 lanes
    edge_index = pd.MultiIndex.from_tuples(
        list(core.edges()),
        names=["source_center", "destination_center"]
    )
    leg_index = pd.MultiIndex.from_frame(
        legs[["source_center", "destination_center"]]
    )
    core_legs = legs.loc[leg_index.isin(edge_index)].copy()

    both_located = (
        core_legs["source_center"].isin(located_ids)
        & core_legs["destination_center"].isin(located_ids)
    )

    largest_strong = max(
        len(group) for group in nx.strongly_connected_components(core)
    )

    core_candidates[size] = {
        "graph": core,
        "hub_ids": core_ids,
        "legs": core_legs
    }

    results.append({
        "hubs": size,
        "directed_lanes": core.number_of_edges(),
        "journey_legs": len(core_legs),
        "all_legs_kept_pct": round(100 * len(core_legs) / len(legs), 2),
        "weakly_connected": nx.is_weakly_connected(core),
        "largest_strong_group_pct": round(100 * largest_strong / size, 2),
        "located_hubs": len(core_ids & located_ids),
        "both_ends_located_pct": round(100 * both_located.mean(), 2)
    })

core_comparison = pd.DataFrame(results)
display(core_comparison)

,hubs,directed_lanes,journey_legs,all_legs_kept_pct,weakly_connected,largest_strong_group_pct,located_hubs,both_ends_located_pct
0,50,173,5680,21.54,True,92.00,47,91.13
1,100,292,7907,29.99,True,89.00,91,87.35
2,150,398,9403,35.66,True,83.33,136,87.10


In [3]:
# Prepare a training-only dataset without changing the original legs
split_legs = legs.copy()

split_legs["trip_created"] = pd.to_datetime(
    split_legs["trip_creation_time"],
    errors="coerce"
)

assert split_legs["trip_created"].notna().all(), \
    "Some trip creation dates are missing or invalid."

# Each trip must have one creation timestamp
dates_per_trip = split_legs.groupby("trip_uuid")["trip_created"].nunique()
assert dates_per_trip.eq(1).all(), \
    "Some trips have conflicting creation timestamps."

train_start = split_legs["trip_created"].min().normalize()
train_cutoff = train_start + pd.Timedelta(days=14)

train_legs = split_legs.loc[
    split_legs["trip_created"] < train_cutoff
].copy()

later_legs = split_legs.loc[
    split_legs["trip_created"] >= train_cutoff
].copy()

assert set(train_legs["trip_uuid"]).isdisjoint(
    set(later_legs["trip_uuid"])
)

# Recalculate lane counts using training data ONLY
train_lanes = (
    train_legs.groupby(
        ["source_center", "destination_center"],
        as_index=False
    )
    .agg(
        leg_count=("trip_uuid", "size"),
        trip_count=("trip_uuid", "nunique")
    )
)

train_lanes_3 = train_lanes.loc[
    train_lanes["trip_count"] >= 3
].copy()

print("Training starts:", train_start)
print("Training ends before:", train_cutoff)
print("Training journey legs:", len(train_legs))
print("Training distinct trips:", train_legs["trip_uuid"].nunique())
print("Later-period journey legs:", len(later_legs))
print("Training lanes with at least 3 trips:", len(train_lanes_3))

Training starts: 2018-09-12 00:00:00
Training ends before: 2018-09-26 00:00:00
Training journey legs: 17674
Training distinct trips: 9969
Later-period journey legs: 8695
Training lanes with at least 3 trips: 1726


In [4]:
# Build the graph using training-only lane counts
train_graph = nx.from_pandas_edgelist(
    train_lanes_3,
    source="source_center",
    target="destination_center",
    edge_attr=["trip_count", "leg_count"],
    create_using=nx.DiGraph()
)

largest_group = max(
    nx.weakly_connected_components(train_graph),
    key=lambda group: (len(group), tuple(sorted(group)))
)

H_train = train_graph.subgraph(largest_group).copy()
U_train = H_train.to_undirected()

traffic_train = {
    hub: (
        sum(d["leg_count"] for _, _, d
            in H_train.in_edges(hub, data=True))
        + sum(d["leg_count"] for _, _, d
              in H_train.out_edges(hub, data=True))
    )
    for hub in H_train
}

seed = sorted(
    H_train,
    key=lambda hub: (-traffic_train[hub], hub)
)[0]

selected = {seed}
train_selection_order = [seed]

while len(train_selection_order) < min(150, len(H_train)):
    candidates = set()

    for hub in selected:
        candidates.update(U_train.neighbors(hub))

    candidates -= selected

    def added_training_legs(hub):
        incoming = sum(
            H_train[other][hub]["leg_count"]
            for other in H_train.predecessors(hub)
            if other in selected
        )
        outgoing = sum(
            H_train[hub][other]["leg_count"]
            for other in H_train.successors(hub)
            if other in selected
        )
        return incoming + outgoing

    chosen = sorted(
        candidates,
        key=lambda hub: (
            -added_training_legs(hub),
            -traffic_train[hub],
            hub
        )
    )[0]

    selected.add(chosen)
    train_selection_order.append(chosen)

located_ids = set(hubs.loc[hubs["located"], "hub"])
train_core_candidates = {}
comparison_rows = []

training_leg_pairs = pd.MultiIndex.from_frame(
    train_legs[["source_center", "destination_center"]]
)

for size in [50, 100, 150]:
    if size > len(train_selection_order):
        continue

    hub_ids = set(train_selection_order[:size])
    core_graph = H_train.subgraph(hub_ids).copy()

    edge_pairs = pd.MultiIndex.from_tuples(
        list(core_graph.edges()),
        names=["source_center", "destination_center"]
    )

    selected_legs = train_legs.loc[
        training_leg_pairs.isin(edge_pairs)
    ].copy()

    both_located = (
        selected_legs["source_center"].isin(located_ids)
        & selected_legs["destination_center"].isin(located_ids)
    )

    largest_strong = max(
        len(group)
        for group in nx.strongly_connected_components(core_graph)
    )

    train_core_candidates[size] = {
        "graph": core_graph,
        "hub_ids": hub_ids,
        "train_legs": selected_legs
    }

    comparison_rows.append({
        "hubs": size,
        "directed_lanes": core_graph.number_of_edges(),
        "training_legs": len(selected_legs),
        "training_legs_kept_pct": round(
            100 * len(selected_legs) / len(train_legs), 2
        ),
        "weakly_connected": nx.is_weakly_connected(core_graph),
        "largest_strong_group_pct": round(
            100 * largest_strong / size, 2
        ),
        "located_hubs": len(hub_ids & located_ids),
        "both_ends_located_pct": round(
            100 * both_located.mean(), 2
        )
    })

train_core_comparison = pd.DataFrame(comparison_rows)
display(train_core_comparison)

,hubs,directed_lanes,training_legs,training_legs_kept_pct,weakly_connected,largest_strong_group_pct,located_hubs,both_ends_located_pct
0,50,165,3828,21.66,True,92.00,47,89.32
1,100,292,5395,30.53,True,93.00,92,87.08
2,150,387,6418,36.31,True,81.33,134,86.26


In [5]:
FINAL_CORE_SIZE = 100

final_core = train_core_candidates[FINAL_CORE_SIZE]
final_graph = final_core["graph"]
final_ids = final_core["hub_ids"]

final_edges = pd.MultiIndex.from_tuples(
    list(final_graph.edges()),
    names=["source_center", "destination_center"]
)

later_pairs = pd.MultiIndex.from_frame(
    later_legs[["source_center", "destination_center"]]
)

# Later legs using the exact directed lanes frozen from training
on_frozen_lanes = later_pairs.isin(final_edges)

# Later legs between core hubs, including lanes absent from the core
within_core_hubs = (
    later_legs["source_center"].isin(final_ids)
    & later_legs["destination_center"].isin(final_ids)
)

print("Frozen core hubs:", len(final_ids))
print("Frozen directed lanes:", final_graph.number_of_edges())
print("Core training legs:", len(final_core["train_legs"]))
print("Total later-period legs:", len(later_legs))

print(
    "Later legs on frozen lanes:",
    int(on_frozen_lanes.sum()),
    f"({100 * on_frozen_lanes.mean():.2f}%)"
)

print(
    "Later legs between core hubs:",
    int(within_core_hubs.sum()),
    f"({100 * within_core_hubs.mean():.2f}%)"
)

print(
    "Later legs between core hubs on other lanes:",
    int((within_core_hubs & ~on_frozen_lanes).sum())
)

print("\nCore hubs needing location review:")
display(
    hubs.loc[
        hubs["hub"].isin(final_ids) & ~hubs["located"],
        ["hub", "name", "search_city", "search_state", "match"]
    ]
)

Frozen core hubs: 100
Frozen directed lanes: 292
Core training legs: 5395
Total later-period legs: 8695
Later legs on frozen lanes: 2422 (27.86%)
Later legs between core hubs: 2459 (28.28%)
Later legs between core hubs on other lanes: 37

Core hubs needing location review:


,hub,name,search_city,search_state,match
5,IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),Chandigarh,Punjab,no_exact_match
45,IND533106AAA,Rajamundry_AtoNgrRd_I (Andhra Pradesh),Rajamundry,Andhra Pradesh,no_exact_match
53,IND403726AAA,Goa_Hub (Goa),Goa,Goa,no_exact_match
76,IND151001AAA,Bhatinda_DPC (Punjab),Bhatinda,Punjab,no_exact_match
376,IND122001AAA,GGN_DPC (Haryana),GGN,Haryana,no_exact_match
424,IND560043AAC,HBR Layout PC (Karnataka),HBR Layout PC,Karnataka,no_exact_match
928,IND121102AAA,Palwal (Haryana),Palwal,Haryana,ambiguous_city_state
1176,IND462011AAA,Bhopal MP Nagar (Madhya Pradesh),Bhopal,Madhya Pradesh,ambiguous_city_state


In [6]:
# Hub records for the frozen core
core_nodes = hubs.loc[hubs["hub"].isin(final_ids)].copy()

# Lane counts come ONLY from the training period
training_lane_pairs = pd.MultiIndex.from_frame(
    train_lanes_3[["source_center", "destination_center"]]
)

core_edges = train_lanes_3.loc[
    training_lane_pairs.isin(final_edges)
].copy()

core_train_legs = final_core["train_legs"].copy()
core_later_legs = later_legs.loc[on_frozen_lanes].copy()

assert len(core_nodes) == 100
assert len(core_edges) == 292
assert len(core_train_legs) == 5395
assert len(core_later_legs) == 2422
assert int(core_edges["leg_count"].sum()) == len(core_train_legs)

core_nodes.to_csv(PROCESSED / "core_nodes.csv", index=False)
core_edges.to_csv(PROCESSED / "core_edges.csv", index=False)
core_train_legs.to_csv(
    PROCESSED / "core_train_legs.csv", index=False
)
core_later_legs.to_csv(
    PROCESSED / "core_later_legs.csv", index=False
)
train_core_comparison.to_csv(
    PROCESSED / "core_training_comparison.csv", index=False
)

core_review = core_nodes.loc[~core_nodes["located"]].copy()
core_review.to_csv(
    PROCESSED / "core_location_review.csv", index=False
)

print("Saved core nodes, edges, training legs, later legs,")
print("comparison table and location review.")
print("Core hubs still needing location review:", len(core_review))
print("Capacity estimates have not been added yet.")

Saved core nodes, edges, training legs, later legs,
comparison table and location review.
Core hubs still needing location review: 8
Capacity estimates have not been added yet.


In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"

core_review = pd.read_csv(
    PROCESSED / "core_location_review.csv"
)

print("Hubs needing review:", len(core_review))

Hubs needing review: 8


In [3]:
import requests
import time

# Search alternatives only; these are not approved corrections
search_options = {
    "IND160002AAC": ["Chandigarh", "Mehmadpur"],
    "IND533106AAA": ["Rajahmundry", "Rajamahendravaram"],
    "IND403726AAA": ["Goa"],  # State-level name: needs more evidence
    "IND151001AAA": ["Bathinda"],
    "IND122001AAA": ["Gurugram"],
    "IND560043AAC": ["HBR Layout", "Bengaluru"],
    "IND121102AAA": ["Palwal"],
    "IND462011AAA": ["Bhopal"],
}

candidate_rows = []
search_errors = []

for _, hub_row in core_review.iterrows():
    hub_id = hub_row["hub"]
    print("Checking:", hub_row["name"])

    for query in search_options.get(hub_id, []):
        try:
            response = requests.get(
                "https://geocoding-api.open-meteo.com/v1/search",
                params={
                    "name": query,
                    "count": 100,
                    "language": "en",
                    "format": "json",
                    "countryCode": "IN",
                },
                timeout=(15, 60),
            )
            response.raise_for_status()

            for place in response.json().get("results", []):
                if place.get("country_code") != "IN":
                    continue

                candidate_rows.append({
                    "hub": hub_id,
                    "hub_name": hub_row["name"],
                    "expected_state": hub_row["search_state"],
                    "query": query,
                    "candidate_id": place.get("id"),
                    "candidate_name": place.get("name"),
                    "candidate_state": place.get("admin1"),
                    "district": place.get("admin2"),
                    "latitude": place.get("latitude"),
                    "longitude": place.get("longitude"),
                })

        except (requests.RequestException, ValueError) as error:
            search_errors.append({
                "hub": hub_id,
                "query": query,
                "error": str(error),
            })

        time.sleep(0.5)

candidates = pd.DataFrame(candidate_rows)
errors = pd.DataFrame(search_errors)

candidates.to_csv(
    PROCESSED / "core_location_candidates.csv", index=False
)

print("\nLocation candidates:")
display(candidates)

if not errors.empty:
    print("\nSearch errors:")
    display(errors)

print("\nExisting hub coordinates have NOT been changed.")

Checking: Chandigarh_Mehmdpur_H (Punjab)
Checking: Rajamundry_AtoNgrRd_I (Andhra Pradesh)
Checking: Goa_Hub (Goa)
Checking: Bhatinda_DPC (Punjab)
Checking: GGN_DPC (Haryana)
Checking: HBR Layout PC (Karnataka)
Checking: Palwal (Haryana)
Checking: Bhopal MP Nagar (Madhya Pradesh)

Location candidates:


,hub,hub_name,expected_state,query,candidate_id,candidate_name,candidate_state,district,latitude,longitude
0,IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),Punjab,Chandigarh,1274746,Chandigarh,Chandigarh,Chandigarh District,30.73629,76.78840
1,IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),Punjab,Chandigarh,13494735,Chandigarh,Tripura,Sepahijala,23.50211,91.33715
2,IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),Punjab,Chandigarh,9915976,Chandīgarh,Haryana,Nuh,30.72778,76.77778
3,IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),Punjab,Chandigarh,10553678,Chandigarhi,Uttar Pradesh,Unao District,26.43376,80.78706
4,IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),Punjab,Chandigarh,7668602,Chandigarh International Airport,Chandigarh,Chandigarh District,30.67350,76.78850
...,...,...,...,...,...,...,...,...,...,...
154,IND462011AAA,Bhopal MP Nagar (Madhya Pradesh),Madhya Pradesh,Bhopal,10481385,Bhopāl Nagla,Uttar Pradesh,Alīgarh District,27.71964,78.01261
155,IND462011AAA,Bhopal MP Nagar (Madhya Pradesh),Madhya Pradesh,Bhopal,10446175,Bhopāl Tegnūr,Karnataka,Kalaburagi,17.38567,76.91820
156,IND462011AAA,Bhopal MP Nagar (Madhya Pradesh),Madhya Pradesh,Bhopal,10440082,Bhopal Nagalia,Uttar Pradesh,Alīgarh District,28.08111,77.93857
157,IND462011AAA,Bhopal MP Nagar (Madhya Pradesh),Madhya Pradesh,Bhopal,9975146,Bhopālrām ki Dhāni,Rajasthan,Bīkāner District,28.41656,73.76629



Existing hub coordinates have NOT been changed.


In [4]:
import unicodedata

def normalise(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return " ".join(text.casefold().split())

same_name = (
    candidates["candidate_name"].map(normalise)
    == candidates["query"].map(normalise)
)

same_state = (
    candidates["candidate_state"].map(normalise)
    == candidates["expected_state"].map(normalise)
)

review_candidates = candidates.loc[
    same_name & same_state,
    [
        "hub", "hub_name", "query", "candidate_id",
        "candidate_name", "candidate_state", "district",
        "latitude", "longitude"
    ]
].drop_duplicates()

print(review_candidates.to_string(index=False))

         hub                               hub_name       query  candidate_id candidate_name candidate_state            district  latitude  longitude
IND533106AAA Rajamundry_AtoNgrRd_I (Andhra Pradesh) Rajahmundry       1258932    Rajahmundry  Andhra Pradesh       East Godavari  17.00517   81.77784
IND533106AAA Rajamundry_AtoNgrRd_I (Andhra Pradesh) Rajahmundry      10791535    Rajahmundry  Andhra Pradesh               Eluru  17.18144   81.36903
IND533106AAA Rajamundry_AtoNgrRd_I (Andhra Pradesh) Rajahmundry       8439080    Rajahmundry  Andhra Pradesh       East Godavari  17.10093   81.88110
IND151001AAA                  Bhatinda_DPC (Punjab)    Bathinda       1276070       Bathinda          Punjab            Bathinda  30.20747   74.93893
IND122001AAA                      GGN_DPC (Haryana)    Gurugram       1270642       Gurugram         Haryana            Gurugram  28.46010   77.02635
IND560043AAC              HBR Layout PC (Karnataka)   Bengaluru       1277333      Bengaluru       K

In [5]:
core_nodes = pd.read_csv(PROCESSED / "core_nodes.csv")

# Explicit city-level approximations, not verified warehouse coordinates
city_proxies = {
    "IND151001AAA": (30.20747, 74.93893, 1276070, "Bathinda"),
    "IND122001AAA": (28.46010, 77.02635, 1270642, "Gurugram"),
    "IND560043AAC": (12.97194, 77.59369, 1277333, "Bengaluru"),
}

# Back up once before making changes
backup = PROCESSED / "core_nodes_before_location_review.csv"
if not backup.exists():
    core_nodes.to_csv(backup, index=False)

for hub_id, (lat, lon, place_id, city) in city_proxies.items():
    mask = core_nodes["hub"].eq(hub_id)

    core_nodes.loc[mask, "latitude"] = lat
    core_nodes.loc[mask, "longitude"] = lon
    core_nodes.loc[mask, "match"] = "reviewed_city_proxy"
    core_nodes.loc[mask, "coordinate_method"] = "city_approximation"
    core_nodes.loc[mask, "review_evidence"] = (
        f"Open-Meteo candidate {place_id}: {city}; "
        "city alias/locality interpretation, warehouse position unverified"
    )

# Include both earlier accepted matches and the explicit city proxies
core_nodes["located"] = (
    core_nodes["match"].isin(["city+state", "reviewed_city_proxy"])
    & pd.to_numeric(core_nodes["latitude"], errors="coerce").between(-90, 90)
    & pd.to_numeric(core_nodes["longitude"], errors="coerce").between(-180, 180)
)

core_review = core_nodes.loc[~core_nodes["located"]].copy()

core_nodes.to_csv(PROCESSED / "core_nodes.csv", index=False)
core_review.to_csv(
    PROCESSED / "core_location_review.csv", index=False
)

print("Core hubs with approximate locations:",
      int(core_nodes["located"].sum()))
print("Still unresolved:", len(core_review))

Core hubs with approximate locations: 95
Still unresolved: 5


In [6]:
core_train_legs = pd.read_csv(PROCESSED / "core_train_legs.csv")
core_later_legs = pd.read_csv(PROCESSED / "core_later_legs.csv")

located_ids = set(
    core_nodes.loc[core_nodes["located"], "hub"]
)

coverage_rows = []

for period, frame in [
    ("training", core_train_legs),
    ("later", core_later_legs),
]:
    both = (
        frame["source_center"].isin(located_ids)
        & frame["destination_center"].isin(located_ids)
    )

    coverage_rows.append({
        "period": period,
        "core_legs": len(frame),
        "both_ends_located": int(both.sum()),
        "both_ends_located_pct": round(100 * both.mean(), 2),
    })

location_coverage = pd.DataFrame(coverage_rows)
location_coverage.to_csv(
    PROCESSED / "core_location_coverage.csv", index=False
)

DOCS = ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)

section = f"""
## Core network selection and location limitations

- Split by trip_creation_time, keeping whole trips together.
- Training window: 12–25 September 2018.
- Later evaluation period: 26 September onward.
- This split must also be used by Person B.
- Lane eligibility requires at least 3 distinct training trips.
- Selected a traffic-based connected core using training data only.
- Selection is a heuristic, not a guaranteed optimal solution.
- Frozen core: 100 hubs and 292 directed lanes.
- Retained training legs: 5,395 (30.53% of all training legs).
- Retained later legs: 2,422 (27.86% of all later legs).
- Largest strongly connected group: 93 of the 100 hubs.
- The 37 later legs between core hubs on other lanes remain excluded.
- Approximate locations available: 95 hubs; unresolved: 5.
- Coordinates represent city approximations, not verified warehouses.
- Three reviewed city proxies are explicitly labelled in core_nodes.csv.
- Postal validation remains incomplete.
- Unresolved locations remain missing; missing rainfall must not become zero.
- Weather coverage and findings apply to the covered subset.
- Daily historical weather is retrospective information, not a forecast
  available when a trip began.
- Capacity estimates have not yet been calculated.

### Journey-leg location coverage

{location_coverage.to_string(index=False)}
"""

# Update only our section, preserving other documentation
assumptions_file = DOCS / "assumptions.md"
existing = (
    assumptions_file.read_text(encoding="utf-8")
    if assumptions_file.exists()
    else "# SupplyShield assumptions\n"
)

start_marker = "<!-- CORE_SELECTION_START -->"
end_marker = "<!-- CORE_SELECTION_END -->"
new_section = f"{start_marker}\n{section}\n{end_marker}"

if start_marker in existing and end_marker in existing:
    before, remainder = existing.split(start_marker, 1)
    _, after = remainder.split(end_marker, 1)
    updated = before + new_section + after
else:
    updated = existing.rstrip() + "\n\n" + new_section + "\n"

assumptions_file.write_text(updated, encoding="utf-8")

display(location_coverage)
print("Saved coverage and docs/assumptions.md")

,period,core_legs,both_ends_located,both_ends_located_pct
0,training,5395,4886,90.57
1,later,2422,2265,93.52


Saved coverage and docs/assumptions.md


In [7]:
core_all_legs = pd.concat(
    [core_train_legs, core_later_legs],
    ignore_index=True
)

for column in ["od_start_time", "od_end_time"]:
    core_all_legs[column] = pd.to_datetime(
        core_all_legs[column],
        errors="coerce"
    )

assert core_all_legs[
    ["od_start_time", "od_end_time"]
].notna().all().all(), "Missing or invalid journey dates."

assert (
    core_all_legs["od_end_time"] >= core_all_legs["od_start_time"]
).all(), "Some journeys end before they start."

weather_start = core_all_legs["od_start_time"].min().date()
weather_end = core_all_legs["od_end_time"].max().date()

weather_hubs = core_nodes.loc[
    core_nodes["located"],
    ["hub", "latitude", "longitude"]
].copy()

weather_hubs.to_csv(
    PROCESSED / "weather_hubs.csv", index=False
)

print("Hubs prepared for weather:", len(weather_hubs))
print("Weather start date:", weather_start)
print("Weather end date:", weather_end)
print("Core journey legs:", len(core_all_legs))

Hubs prepared for weather: 95
Weather start date: 2018-09-12
Weather end date: 2018-10-07
Core journey legs: 7817


In [8]:
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

weather_hubs = pd.read_csv(PROCESSED / "weather_hubs.csv")

start_date = "2018-09-12"
end_date = "2018-10-07"
expected_dates = set(
    pd.date_range(start_date, end_date).strftime("%Y-%m-%d")
)

cache_dir = PROCESSED / "weather_cache_era5"
cache_dir.mkdir(parents=True, exist_ok=True)

session = requests.Session()
retry = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retry))

weather_tables = []
failures = []

for number, row in enumerate(
    weather_hubs.itertuples(index=False), start=1
):
    cache_file = cache_dir / f"{row.hub}_{start_date}_{end_date}.csv"

    try:
        table = None

        if cache_file.exists():
            saved = pd.read_csv(cache_file, dtype={"date": str})
            required = {
                "date", "rain_mm", "precipitation_mm",
                "temperature_mean_c", "requested_latitude",
                "requested_longitude"
            }

            if (
                required.issubset(saved.columns)
                and len(saved) == len(expected_dates)
                and set(saved["date"]) == expected_dates
                and saved["requested_latitude"].eq(row.latitude).all()
                and saved["requested_longitude"].eq(row.longitude).all()
                and saved[
                    ["rain_mm", "precipitation_mm", "temperature_mean_c"]
                ].notna().all().all()
            ):
                table = saved

        if table is None:
            response = session.get(
                "https://archive-api.open-meteo.com/v1/archive",
                params={
                    "latitude": row.latitude,
                    "longitude": row.longitude,
                    "start_date": start_date,
                    "end_date": end_date,
                    "daily": (
                        "rain_sum,precipitation_sum,"
                        "temperature_2m_mean"
                    ),
                    "timezone": "Asia/Kolkata",
                    "models": "era5",
                },
                timeout=(15, 60),
            )
            response.raise_for_status()
            payload = response.json()

            if payload.get("error"):
                raise ValueError(payload.get("reason"))

            daily = payload["daily"]
            table = pd.DataFrame({
                "date": daily["time"],
                "rain_mm": daily["rain_sum"],
                "precipitation_mm": daily["precipitation_sum"],
                "temperature_mean_c": daily["temperature_2m_mean"],
            })

            if (
                len(table) != len(expected_dates)
                or set(table["date"]) != expected_dates
            ):
                raise ValueError("Unexpected dates returned.")

            if table[
                ["rain_mm", "precipitation_mm", "temperature_mean_c"]
            ].isna().any().any():
                raise ValueError("Weather response contains missing values.")

            table["hub"] = row.hub
            table["requested_latitude"] = row.latitude
            table["requested_longitude"] = row.longitude
            table["grid_latitude"] = payload["latitude"]
            table["grid_longitude"] = payload["longitude"]
            table["weather_model"] = "era5"
            table["timezone"] = "Asia/Kolkata"

            temporary = cache_file.with_suffix(".tmp")
            table.to_csv(temporary, index=False)
            temporary.replace(cache_file)

            time.sleep(1)

        weather_tables.append(table)

    except (
        requests.RequestException, ValueError, KeyError, OSError
    ) as error:
        failures.append({"hub": row.hub, "error": str(error)})
        print("Failed:", row.hub, str(error)[:120])

    if number % 10 == 0 or number == len(weather_hubs):
        print(f"Processed {number}/{len(weather_hubs)} hubs")

session.close()

if weather_tables:
    weather_daily = pd.concat(weather_tables, ignore_index=True)
    assert not weather_daily.duplicated(["hub", "date"]).any()
    weather_daily.to_csv(
        PROCESSED / "weather_daily.csv", index=False
    )

pd.DataFrame(failures, columns=["hub", "error"]).to_csv(
    PROCESSED / "weather_download_errors.csv", index=False
)

print("\nSuccessful hubs:", len(weather_tables))
print("Failed hubs:", len(failures))
print("Weather rows:", sum(len(t) for t in weather_tables))

Processed 10/95 hubs
Processed 20/95 hubs
Processed 30/95 hubs
Processed 40/95 hubs
Processed 50/95 hubs
Processed 60/95 hubs
Processed 70/95 hubs
Processed 80/95 hubs
Processed 90/95 hubs
Processed 95/95 hubs

Successful hubs: 95
Failed hubs: 0
Weather rows: 2470


In [9]:
weather = pd.read_csv(PROCESSED / "weather_daily.csv")
weather["date"] = pd.to_datetime(weather["date"]).dt.normalize()

assert not weather.duplicated(["hub", "date"]).any()

train = pd.read_csv(PROCESSED / "core_train_legs.csv")
later = pd.read_csv(PROCESSED / "core_later_legs.csv")

train["split"] = "training"
later["split"] = "later"

enriched = pd.concat([train, later], ignore_index=True)
original_count = len(enriched)

enriched["departure_date"] = pd.to_datetime(
    enriched["od_start_time"], errors="raise"
).dt.normalize()

enriched["arrival_date"] = pd.to_datetime(
    enriched["od_end_time"], errors="raise"
).dt.normalize()

variables = ["rain_mm", "precipitation_mm", "temperature_mean_c"]

source_weather = weather[["hub", "date"] + variables].rename(
    columns={
        "hub": "source_center",
        "date": "departure_date",
        **{v: f"source_{v}" for v in variables},
    }
)

destination_weather = weather[["hub", "date"] + variables].rename(
    columns={
        "hub": "destination_center",
        "date": "arrival_date",
        **{v: f"destination_{v}" for v in variables},
    }
)

enriched = enriched.merge(
    source_weather,
    on=["source_center", "departure_date"],
    how="left",
    validate="many_to_one",
)

enriched = enriched.merge(
    destination_weather,
    on=["destination_center", "arrival_date"],
    how="left",
    validate="many_to_one",
)

assert len(enriched) == original_count, "Join changed the leg count."

enriched["source_weather_available"] = enriched[
    [f"source_{v}" for v in variables]
].notna().all(axis=1)

enriched["destination_weather_available"] = enriched[
    [f"destination_{v}" for v in variables]
].notna().all(axis=1)

enriched["both_weather_available"] = (
    enriched["source_weather_available"]
    & enriched["destination_weather_available"]
)

# Keep missing weather as missing—do not fill with zero
enriched.to_csv(PROCESSED / "legs_enriched.csv", index=False)

summary = (
    enriched.groupby("split", as_index=False)
    .agg(
        journey_legs=("trip_uuid", "size"),
        source_weather_available=("source_weather_available", "sum"),
        destination_weather_available=("destination_weather_available", "sum"),
        both_weather_available=("both_weather_available", "sum"),
    )
)

summary["both_weather_pct"] = (
    100 * summary["both_weather_available"] / summary["journey_legs"]
).round(2)

summary.to_csv(
    PROCESSED / "weather_join_coverage.csv", index=False
)

display(summary)
print("Saved: legs_enriched.csv")

,split,journey_legs,source_weather_available,destination_weather_available,both_weather_available,both_weather_pct
0,later,2422,2368,2319,2265,93.52
1,training,5395,5156,5125,4886,90.57


Saved: legs_enriched.csv


In [10]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
CONFIG = ROOT / "config"
CONFIG.mkdir(exist_ok=True)

locations = pd.read_csv(PROCESSED / "hubs_threshold3.csv")
reviewed_core = pd.read_csv(PROCESSED / "core_nodes.csv")

# Preserve location information only—not capacity or model results
columns = [
    "hub", "name", "search_city", "search_state",
    "match", "latitude", "longitude",
    "coordinate_method", "review_evidence"
]

for column in columns:
    if column not in locations:
        locations[column] = pd.NA

# Overlay the latest reviewed core locations
locations = locations.set_index("hub")

for column in columns[1:]:
    if column in reviewed_core:
        for hub_id, value in reviewed_core.set_index("hub")[column].items():
            if hub_id in locations.index and pd.notna(value):
                locations.loc[hub_id, column] = value

locations.reset_index()[columns].to_csv(
    CONFIG / "hub_location_snapshot.csv", index=False
)

print("Saved config/hub_location_snapshot.csv")

Saved config/hub_location_snapshot.csv
